In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd

Load annotator files

In [3]:
ann1 = pd.read_csv("/content/drive/MyDrive/ShopHallu/Samples_for_Annotator 2.csv")
ann2 = pd.read_csv("/content/drive/MyDrive/ShopHallu/Annotator 2.csv")

In [4]:
print("Annotator 1 rows:", len(ann1))
print("Annotator 2 rows:", len(ann2))

Annotator 1 rows: 500
Annotator 2 rows: 500


In [5]:
print("\nColumns:")
print(ann1.columns.tolist())


Columns:
['asin', 'category', 'prompt_type', 'model', 'temperature', 'influencer_review', 'evidence_review', 'supported', 'fabricated_feature', 'exaggerated_performance', 'contradiction', 'emotional_only', 'similarity_score', 'keyword_overlap_count', 'keyword_overlap_ratio', 'unsupported_keywords', 'exaggeration_flag', 'suggested_label', 'annotator_comment']


In [6]:
print(len(ann1) == len(ann2))

True


Cohen's Kappa for Supported

In [7]:
from sklearn.metrics import cohen_kappa_score

kappa = cohen_kappa_score(
    ann1["supported"],
    ann2["supported"]
)

print("Cohen Kappa:", round(kappa, 4))

Cohen Kappa: 0.8278


Raw Agreement

In [8]:
agreement = (
    ann1["supported"] == ann2["supported"]
).mean()

print("Agreement:", round(agreement, 4))

Agreement: 0.89


Confusion Matrix

In [9]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(
    ann1["supported"],
    ann2["supported"]
)

print(cm)

[[170   0  24]
 [  6  88  13]
 [ 10   2 187]]


Find disagreements

In [10]:
disagree = ann1["supported"] != ann2["supported"]

disagreement_rows = pd.DataFrame({
    "ann1": ann1["supported"],
    "ann2": ann2["supported"]
})[disagree]

print("Total disagreements:", len(disagreement_rows))

Total disagreements: 55


Save disagreement rows and final consensus

In [11]:
mask = ann1["supported"] != ann2["supported"]

compare = pd.DataFrame({
    "row_id": ann1.index,
    "ann1": ann1["supported"],
    "ann2": ann2["supported"]
})

compare = compare[mask]

In [12]:
mask = ann1["supported"] != ann2["supported"]

disagree = ann1.loc[mask].copy()

disagree["row_id"] = disagree.index

disagree["ann1_label"] = ann1.loc[mask, "supported"]
disagree["ann2_label"] = ann2.loc[mask, "supported"]

cols = ["row_id", "ann1_label", "ann2_label"] + [
    c for c in disagree.columns
    if c not in ["row_id", "ann1_label", "ann2_label"]
]

disagree = disagree[cols]

disagree.to_excel(
    "disagreement_cases.xlsx",
    index=False
)

In [13]:
import os

print(os.listdir())

['.config', 'disagreement_cases.xlsx', 'drive', 'sample_data']


In [14]:
disagree_idx = ann1.index[
    ann1["supported"] != ann2["supported"]
]

print(disagree_idx.tolist())

[3, 31, 52, 53, 54, 81, 88, 104, 114, 118, 119, 122, 123, 129, 145, 146, 158, 181, 206, 216, 221, 227, 228, 234, 239, 250, 253, 269, 271, 280, 295, 301, 305, 306, 314, 319, 326, 330, 337, 348, 371, 386, 387, 392, 397, 407, 409, 414, 427, 453, 459, 475, 487, 490, 499]


disagreement row index

In [15]:
excel_rows = [i + 2 for i in disagree_idx]

print(excel_rows)

[5, 33, 54, 55, 56, 83, 90, 106, 116, 120, 121, 124, 125, 131, 147, 148, 160, 183, 208, 218, 223, 229, 230, 236, 241, 252, 255, 271, 273, 282, 297, 303, 307, 308, 316, 321, 328, 332, 339, 350, 373, 388, 389, 394, 399, 409, 411, 416, 429, 455, 461, 477, 489, 492, 501]


In [16]:
import os

for f in os.listdir('/content/drive/MyDrive/ShopHallu'):
    print(repr(f))

'04.Evaluation.ipynb'
'All_Beauty.json'
'Cell_Phones_and_Accessories.json'
'Grocery_and_Gourmet_Food.json'
'01.ShopHallu_dataset_generated_for_rule_based_analysis.gsheet'
'01.ShopHallu_dataset_generated_for_rule_based_analysis.csv'
'01.ShopHallu_dataset_generated_for_rule_based_analysis.xlsx'
'All Figures'
'01.ShopHallu_dataset_generated_for_rule_based_analysis_with_generation_text.csv'
'01.ShopHallu_dataset_generated_for_rule_based_analysis_with_generation_text.xlsx'
'02.Rule_Based_Analysis_for_Annotation.xlsx'
'02.Rule_Based_Analysis_for_Annotation.csv'
'01.ShopHallu_Dataset_Generation.ipynb'
'02.Rule_Based_Analysis_for_Annotation.ipynb'
'Annotator 1 (Main Dataset).csv'
'Annotator 1 (Main Dataset).xlsx'
'Dataset (Statistical Analysis).csv'
'Dataset (Statistical Analysis).xlsx'
'03.Samples_from_main_dataset_for_Cohen_s_Kappa.ipynb'
'Samples_for_Annotator 2.csv'
'Samples_for_Annotator 2.xlsx'
'Annotator 2.csv'
'Annotator 2.xlsx'


f1

In [17]:
df = pd.read_csv("/content/drive/MyDrive/ShopHallu/Annotator 1 (Main Dataset).csv")

print(df["weak_annotation_signal"].value_counts())
print()
print(df["weak_annotation_signal"].dtype)

weak_annotation_signal
Potential_Hallucination    13560
Review_Needed               8469
Likely_Supported             246
Name: count, dtype: int64

object


In [18]:
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix

df = pd.read_csv("/content/drive/MyDrive/ShopHallu/Annotator 1 (Main Dataset).csv")

mapping = {
    "Potential_Hallucination": 0,
    "Likely_Supported": 1,
    "Review_Needed": 2
}
df["suggested_numeric"] = df["weak_annotation_signal"].map(mapping)

compare_df = df[
    (df["supported"] != 2) &
    (df["suggested_numeric"] != 2)
].copy()

print("Rows used for comparison:", len(compare_df))
print()

y_true = compare_df["supported"]
y_pred = compare_df["suggested_numeric"]

print("Classification Report:")
print(classification_report(
    y_true,
    y_pred,
    target_names=["Hallucinated", "Supported"],
    digits=4
))

print("Confusion Matrix:")
print(confusion_matrix(y_true, y_pred))

Rows used for comparison: 9794

Classification Report:
              precision    recall  f1-score   support

Hallucinated     0.5762    0.9975    0.7304      5521
   Supported     0.9407    0.0520    0.0985      4273

    accuracy                         0.5849      9794
   macro avg     0.7584    0.5247    0.4144      9794
weighted avg     0.7352    0.5849    0.4547      9794

Confusion Matrix:
[[5507   14]
 [4051  222]]


In [19]:
review_df = df[
    (df["supported"] != 2) &
    (df["suggested_numeric"] == 2)
].copy()

print("Review_Needed rows with verifiable final label:", len(review_df))
print()
print("Their final labels:")
print(review_df["supported"].value_counts())

Review_Needed rows with verifiable final label: 3384

Their final labels:
supported
0    1788
1    1596
Name: count, dtype: int64


In [20]:
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix

df = pd.read_csv("/content/drive/MyDrive/ShopHallu/Annotator 1 (Main Dataset).csv")

mapping = {
    "Potential_Hallucination": 0,
    "Likely_Supported": 1,
    "Review_Needed": 2
}
df["suggested_numeric"] = df["weak_annotation_signal"].map(mapping)

# --- Table 1: Automated signal coverage ---
print("=== Automated Signal Coverage ===")
print(df["weak_annotation_signal"].value_counts())
print(f"Total: {len(df)}")
print()

# --- Table 2: Where Review_Needed landed ---
review_df = df[
    (df["supported"] != 2) &
    (df["suggested_numeric"] == 2)
].copy()
print("=== Review_Needed Resolution by Human Annotators ===")
print(review_df["supported"].value_counts())
print(f"Total ambiguous cases resolved: {len(review_df)}")
print()

# --- Table 3: Classification report (excluding Review_Needed) ---
compare_df = df[
    (df["supported"] != 2) &
    (df["suggested_numeric"] != 2)
].copy()

y_true = compare_df["supported"]
y_pred = compare_df["suggested_numeric"]

print("=== Automated vs Human Label (Verifiable, Confident Rows Only) ===")
print(f"Rows compared: {len(compare_df)}")
print()
print(classification_report(
    y_true, y_pred,
    target_names=["Hallucinated", "Supported"],
    digits=4
))

# --- Table 4: Full dataset breakdown ---
print("=== Final Label Distribution (Full Dataset) ===")
label_map = {0: "Hallucinated", 1: "Supported", 2: "Neutral"}
print(df["supported"].map(label_map).value_counts())

=== Automated Signal Coverage ===
weak_annotation_signal
Potential_Hallucination    13560
Review_Needed               8469
Likely_Supported             246
Name: count, dtype: int64
Total: 22275

=== Review_Needed Resolution by Human Annotators ===
supported
0    1788
1    1596
Name: count, dtype: int64
Total ambiguous cases resolved: 3384

=== Automated vs Human Label (Verifiable, Confident Rows Only) ===
Rows compared: 9794

              precision    recall  f1-score   support

Hallucinated     0.5762    0.9975    0.7304      5521
   Supported     0.9407    0.0520    0.0985      4273

    accuracy                         0.5849      9794
   macro avg     0.7584    0.5247    0.4144      9794
weighted avg     0.7352    0.5849    0.4547      9794

=== Final Label Distribution (Full Dataset) ===
supported
Neutral         9097
Hallucinated    7309
Supported       5869
Name: count, dtype: int64


In [21]:
g3 = df[df["supported"] == 2]
print(f"Human labeled Neutral: {len(g3)}")
print()
print("Automated said for these:")
print(g3["weak_annotation_signal"].value_counts())

Human labeled Neutral: 9097

Automated said for these:
weak_annotation_signal
Review_Needed              5085
Potential_Hallucination    4002
Likely_Supported             10
Name: count, dtype: int64


In [22]:
cross = pd.crosstab(
    df["weak_annotation_signal"],
    df["supported"].map({0:"Hallucinated", 1:"Supported", 2:"Neutral"})
)
print(cross)
print()
print("Row totals:")
print(cross.sum(axis=1))

supported                Hallucinated  Neutral  Supported
weak_annotation_signal                                   
Likely_Supported                   14       10        222
Potential_Hallucination          5507     4002       4051
Review_Needed                    1788     5085       1596

Row totals:
weak_annotation_signal
Likely_Supported             246
Potential_Hallucination    13560
Review_Needed               8469
dtype: int64


Analysis

In [23]:
df = pd.read_csv("/content/drive/MyDrive/ShopHallu/Annotator 1 (Main Dataset).csv")

df["supported"].value_counts()

,count
supported,
2,9097
0,7309
1,5869


In [24]:
df["asin"].nunique()
df["asin"].value_counts()

,count
asin,
B007JZU8N8,1
B00HZ79SQ8,1
B00CKUDPS2,1
B019RPZAI0,1
B01F2YFW3I,1
...,...
B002GWHBXY,1
B0043OGR1K,1
B0040PUPQ0,1


Step 1: Verifiable Dataset (excluding 'Neutral')

> Add blockquote



In [25]:
verifiable_df = df[df["supported"] != 2]

In [26]:
print(len(verifiable_df))

13178


Step 2: Overall Hallucination Rate

In [27]:
hall = (verifiable_df["supported"] == 0).sum()
supp = (verifiable_df["supported"] == 1).sum()

hall_rate = hall/(hall+supp)*100

print(hall_rate)

55.4636515404462


Step 3: Prompt-wise Hallucination Rate

In [28]:
for p in df["prompt_type"].unique():

    temp = df[df["prompt_type"] == p]
    temp = temp[temp["supported"] != 2]

    rate = (
        (temp["supported"] == 0).sum()
        / len(temp)
    ) * 100

    print(p, rate)

strict 34.74101726551563
neutral 55.077767612076855
persuasive 75.48672566371681


Step 4: Category-wise Hallucination Rate

In [29]:
for c in df["category"].unique():

    temp = df[df["category"] == c]
    temp = temp[temp["supported"] != 2]

    rate = (
        (temp["supported"] == 0).sum()
        / len(temp)
    ) * 100

    print(c, rate)

Accessories 64.6107933349913
Beauty 44.829015544041454
Grocery 58.818097876269626


Step 5: Model-wise Hallucination Rate

In [30]:
for m in df["model"].unique():

    temp = df[df["model"] == m]
    temp = temp[temp["supported"] != 2]

    rate = (
        (temp["supported"] == 0).sum()
        / len(temp)
    ) * 100

    print(m, rate)

openai/gpt-4o-mini 48.394548394548394
google/gemini-3.5-flash 67.79698096696565
meta-llama/llama-3.1-8b-instruct 49.43899018232819


Step 6: Temperature-wise Hallucination Rate

In [31]:
for t in sorted(df["temperature"].unique()):

    temp = df[df["temperature"] == t]
    temp = temp[temp["supported"] != 2]

    rate = (
        (temp["supported"] == 0).sum()
        / len(temp)
    ) * 100

    print(t, rate)

0.0 54.9343339587242
0.3 54.06626506024096
0.5 55.597579425113466
0.7 54.570100697134
1.0 58.15279361459521


Step 7: Fine-grained Hallucination Distribution

In [32]:
hall_df = df[df["supported"] == 0]

In [33]:
print(hall_df.columns.tolist())

['asin', 'category', 'generation_reviews', 'generation_text', 'prompt_type', 'model', 'temperature', 'influencer_review', 'evidence_review', 'supported', 'fabricated_feature', 'exaggerated_performance', 'contradiction', 'emotional_only', 'similarity_score', 'keyword_overlap_count', 'keyword_overlap_ratio', 'unsupported_keywords', 'exaggeration_flag', 'weak_annotation_signal', 'annotator_comment']


In [34]:
cols = [
    "fabricated_feature",
    "exaggerated_performance",
    "contradiction",
    "emotional_only"
]

for c in cols:

    count = hall_df[c].sum()

    pct = count/len(hall_df)*100

    print(c, count, pct)

fabricated_feature 1204 16.472841702011216
exaggerated_performance 4406 60.28184430154604
contradiction 306 4.186619236557668
emotional_only 1393 19.05869475988507


Step 8: Chi-Square Test -
Does prompt type affect hallucination?

In [35]:
from scipy.stats import chi2_contingency

temp = df[df["supported"] != 2]

table = pd.crosstab(
    temp["prompt_type"],
    temp["supported"]
)

chi2,p,dof,expected = chi2_contingency(table)

print("Chi2:",chi2)
print("p:",p)

Chi2: 1479.000869726371
p: 0.0


Step 9: Another Chi-Square

In [36]:
from scipy.stats import chi2_contingency
import numpy as np

verifiable = df[df["supported"] != 2]

comparisons = {
    "prompt_type": "prompt_type",
    "category": "category",
    "model": "model"
}

for name, col in comparisons.items():
    table = pd.crosstab(verifiable[col], verifiable["supported"])
    chi2, p, dof, expected = chi2_contingency(table)
    n = table.sum().sum()
    cramers_v = np.sqrt(chi2 / (n * (min(table.shape) - 1)))
    print(f"{name}: Chi2={chi2:.2f}, p={p:.2e}, Cramér's V={cramers_v:.4f}")

prompt_type: Chi2=1479.00, p=0.00e+00, Cramér's V=0.3350
category: Chi2=376.85, p=1.47e-82, Cramér's V=0.1691
model: Chi2=431.92, p=1.62e-94, Cramér's V=0.1810


Step 10: Another Chi-Square

Step 11: Confidence Intervals

In [37]:
from statsmodels.stats.proportion import proportion_confint

hall = (verifiable_df["supported"] == 0).sum()

total = len(verifiable_df)

low,high = proportion_confint(
    hall,
    total,
    alpha=0.05
)

print(low*100)
print(high*100)

54.61508701513098
56.31221606576142


Step 12: Effect Size (Cramer's V)

In [38]:
import numpy as np

n = table.sum().sum()

cramers_v = np.sqrt(
    chi2/(n*(min(table.shape)-1))
)

print(cramers_v)

0.18104092276618974


Step 13: Logistic Regression

In [39]:
import pandas as pd

df = pd.read_excel("/content/drive/MyDrive/ShopHallu/Dataset (Statistical Analysis).xlsx")

df = df[df["supported"] != 2].copy()

0 = Hallucinated
1 = Supported

In [40]:
df["hallucinated"] = (
    df["supported"] == 0
).astype(int)

In [ ]:
!pip install statsmodels

In [ ]:
import statsmodels.formula.api as smf

In [ ]:
model = smf.logit(
    "hallucinated ~ C(prompt_type) + C(category) + C(model) + temperature",
    data=df
).fit()

print(model.summary())

Optimization terminated successfully.
         Current function value: 0.597123
         Iterations 5
                           Logit Regression Results                           
Dep. Variable:           hallucinated   No. Observations:                13178
Model:                          Logit   Df Residuals:                    13170
Method:                           MLE   Df Model:                            7
Date:                Sun, 02 Aug 2026   Pseudo R-squ.:                  0.1310
Time:                        08:03:10   Log-Likelihood:                -7868.9
converged:                       True   LL-Null:                       -9055.5
Covariance Type:            nonrobust   LLR p-value:                     0.000
                                                   coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------------------------------
Intercept                               

In [ ]:
import statsmodels.api as sm
import statsmodels.formula.api as smf

gee_result = smf.gee(
    formula="""
hallucinated ~
C(prompt_type)
+ C(category)
+ C(model)
+ C(temperature)
""",
    groups="asin",
    data=df,
    family=sm.families.Binomial()
).fit()

print(gee_result.summary())

                               GEE Regression Results                              
Dep. Variable:                hallucinated   No. Observations:                13178
Model:                                 GEE   No. clusters:                       99
Method:                        Generalized   Min. cluster size:                  16
                      Estimating Equations   Max. cluster size:                 212
Family:                           Binomial   Mean cluster size:               133.1
Dependence structure:         Independence   Num. iterations:                     2
Date:                     Sun, 02 Aug 2026   Scale:                           1.000
Covariance type:                    robust   Time:                         08:03:21
                                                   coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------------------------------
Intercept         

In [ ]:
import numpy as np

odds_ratios = pd.DataFrame({
    "Variable": gee_result.params.index,
    "Odds_Ratio": np.exp(gee_result.params),
    "P_Value": gee_result.pvalues
})

print(odds_ratios)

                                                                                  Variable  \
Intercept                                                                        Intercept   
C(prompt_type)[T.persuasive]                                  C(prompt_type)[T.persuasive]   
C(prompt_type)[T.strict]                                          C(prompt_type)[T.strict]   
C(category)[T.Beauty]                                                C(category)[T.Beauty]   
C(category)[T.Grocery]                                              C(category)[T.Grocery]   
C(model)[T.meta-llama/llama-3.1-8b-instruct]  C(model)[T.meta-llama/llama-3.1-8b-instruct]   
C(model)[T.openai/gpt-4o-mini]                              C(model)[T.openai/gpt-4o-mini]   
C(temperature)[T.0.3]                                                C(temperature)[T.0.3]   
C(temperature)[T.0.5]                                                C(temperature)[T.0.5]   
C(temperature)[T.0.7]                                       

Prompt × Model

In [ ]:
gee_prompt_model = smf.gee(
    formula="""
hallucinated ~
C(prompt_type) * C(model)
+ C(category)
+ C(temperature)
""",
    groups="asin",
    data=df,
    family=sm.families.Binomial()
).fit()

print(gee_prompt_model.summary())

                               GEE Regression Results                              
Dep. Variable:                hallucinated   No. Observations:                13178
Model:                                 GEE   No. clusters:                       99
Method:                        Generalized   Min. cluster size:                  16
                      Estimating Equations   Max. cluster size:                 212
Family:                           Binomial   Mean cluster size:               133.1
Dependence structure:         Independence   Num. iterations:                     2
Date:                     Sun, 02 Aug 2026   Scale:                           1.000
Covariance type:                    robust   Time:                         08:03:23
                                                                                coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------------------

Prompt × Temperature

In [ ]:
gee_prompt_temp = smf.gee(
    formula="""
hallucinated ~
C(prompt_type) * C(temperature)
+ C(category)
+ C(model)
""",
    groups="asin",
    data=df,
    family=sm.families.Binomial()
).fit()

print(gee_prompt_temp.summary())

                               GEE Regression Results                              
Dep. Variable:                hallucinated   No. Observations:                13178
Model:                                 GEE   No. clusters:                       99
Method:                        Generalized   Min. cluster size:                  16
                      Estimating Equations   Max. cluster size:                 212
Family:                           Binomial   Mean cluster size:               133.1
Dependence structure:         Independence   Num. iterations:                     2
Date:                     Sun, 02 Aug 2026   Scale:                           1.000
Covariance type:                    robust   Time:                         08:03:25
                                                         coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------------------------------------
Interc

In [ ]:
print(df.shape)

(13178, 20)


In [ ]:
print(df["asin"].nunique())

99


In [ ]:
print(df.groupby("asin").size().head(20))

asin
B00006L9LC     48
B000612466    119
B0009TMU0K    137
B000FI4S1E     61
B000FOI48G    194
B000GE5AAA    165
B000GLRREU    203
B000GZW5MA    121
B000H25Y0M     56
B000LQJT5A    127
B000NKJIXM    186
B000URXP6E    163
B000VK88IA    142
B000VV1YOY    125
B000WS1GNK     95
B0010ZBORW    111
B0012Y0ZG2     96
B0013NB7DW     79
B001E95KNS    192
B001ET7FZE    192
dtype: int64


In [ ]:
counts = df.groupby("asin").size()

print(counts.describe())
print(counts.value_counts().sort_index())

count     99.000000
mean     133.111111
std       42.739574
min       16.000000
25%      106.500000
50%      132.000000
75%      165.000000
max      212.000000
dtype: float64
16     1
48     1
51     1
56     1
58     2
      ..
194    1
199    1
203    1
205    1
212    1
Name: count, Length: 71, dtype: int64


In [ ]:
print(counts.head(10))
print(counts.tail(10))

asin
B00006L9LC     48
B000612466    119
B0009TMU0K    137
B000FI4S1E     61
B000FOI48G    194
B000GE5AAA    165
B000GLRREU    203
B000GZW5MA    121
B000H25Y0M     56
B000LQJT5A    127
dtype: int64
asin
B01BBG6OSI    158
B01BNEYGQU    187
B01CGUSLZC    182
B01CQJXY3M    143
B01DC5F1VC    153
B01DLR9IDI    182
B01E7UKR38    168
B01EWCR3TC    158
B01F2YFW3I    128
B01FLKVRC8    152
dtype: int64


SBERT_Evidence_Similarity_Baseline

In [ ]:
from sentence_transformers import SentenceTransformer, util
from sklearn.metrics import precision_score, recall_score, f1_score

In [ ]:
df = pd.read_csv("/content/drive/MyDrive/ShopHallu/Annotator 1 (Main Dataset).csv")

print(df.columns)
print(df.head())

Index(['asin', 'category', 'generation_reviews', 'generation_text',
       'prompt_type', 'model', 'temperature', 'influencer_review',
       'evidence_review', 'supported', 'fabricated_feature',
       'exaggerated_performance', 'contradiction', 'emotional_only',
       'similarity_score', 'keyword_overlap_count', 'keyword_overlap_ratio',
       'unsupported_keywords', 'exaggeration_flag', 'weak_annotation_signal',
       'annotator_comment'],
      dtype='object')
         asin     category                                 generation_reviews  \
0  B007JZU8N8  Accessories  ["good to have, fits very well", "Expecting an...   
1         NaN  Accessories  ["good to have, fits very well", "Expecting an...   
2         NaN  Accessories  ["good to have, fits very well", "Expecting an...   
3         NaN  Accessories  ["good to have, fits very well", "Expecting an...   
4         NaN  Accessories  ["good to have, fits very well", "Expecting an...   

                                     gener

In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
def compute_similarity(text1, text2):
    emb1 = model.encode(text1, convert_to_tensor=True)
    emb2 = model.encode(text2, convert_to_tensor=True)
    score = util.cos_sim(emb1, emb2).item()
    return score

In [ ]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    classification_report,
)

def run_similarity_baseline(df, threshold=0.5):

    y_true = []
    y_pred = []

    for _, row in df.iterrows():

        # Skip Neutral
        if row["supported"] == 2:
            continue

        generated = row["influencer_review"]
        evidence = row["evidence_review"]

        score = compute_similarity(generated, evidence)

        # 1 = Hallucinated
        # 0 = Not Hallucinated (Supported)
        pred = 1 if score <= threshold else 0

        y_pred.append(pred)

        # Ground truth
        # 0 = Hallucinated
        # 1 = Supported
        y_true.append(1 if row["supported"] == 0 else 0)

    print("Precision:", precision_score(y_true, y_pred))
    print("Recall:", recall_score(y_true, y_pred))
    print("F1-score:", f1_score(y_true, y_pred))

    print("\nClassification Report\n")

    print(
        classification_report(
            y_true,
            y_pred,
            target_names=[
                "Not Hallucinated",
                "Hallucinated"
            ],
            digits=4,
            zero_division=0
        )
    )

In [ ]:
small_df = df.sample(200)
run_similarity_baseline(small_df)

Precision: 0.92
Recall: 0.5897435897435898
F1-score: 0.71875

Classification Report

                  precision    recall  f1-score   support

Not Hallucinated     0.5429    0.9048    0.6786        42
    Hallucinated     0.9200    0.5897    0.7188        78

        accuracy                         0.7000       120
       macro avg     0.7314    0.7473    0.6987       120
    weighted avg     0.7880    0.7000    0.7047       120



In [ ]:
run_similarity_baseline(df)

Precision: 0.8497815003641661
Recall: 0.6385278423860993
F1-score: 0.7291617842356066

Classification Report

                  precision    recall  f1-score   support

Not Hallucinated     0.6563    0.8594    0.7442      5869
    Hallucinated     0.8498    0.6385    0.7292      7309

        accuracy                         0.7369     13178
       macro avg     0.7530    0.7490    0.7367     13178
    weighted avg     0.7636    0.7369    0.7359     13178

